In [11]:
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

df = pd.read_csv('weatherAUS.csv')
df = df.drop("Date",axis=1)

# Encode Target Variable
# Y = 1
# N = 0
df["RainTomorrow"] = df["RainTomorrow"].map({"Yes":1,"No":0})

# Data Cleaning
print("\nMissing Values:")
print(df.isnull().sum())

df = df.drop_duplicates()

print("\nDuplicates Removed!")

num_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())




Missing Values:
Location             0
MinTemp           1485
MaxTemp           1261
Rainfall          3261
Evaporation      62790
Sunshine         69835
WindGustDir      10326
WindGustSpeed    10263
WindDir9am       10566
WindDir3pm        4228
WindSpeed9am      1767
WindSpeed3pm      3062
Humidity9am       2654
Humidity3pm       4507
Pressure9am      15065
Pressure3pm      15028
Cloud9am         55888
Cloud3pm         59358
Temp9am           1767
Temp3pm           3609
RainToday         3261
RainTomorrow      3267
dtype: int64

Duplicates Removed!


In [12]:
X = df.drop("RainTomorrow", axis=1)
y = df["RainTomorrow"]

X = pd.get_dummies(X)
encoded_cols = X.columns.to_list()

numeric_cols = ['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine',
       'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am',
       'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm',
       'Temp9am', 'Temp3pm']

Scaler = StandardScaler()
X[numeric_cols] = Scaler.fit_transform(X[numeric_cols])

# ===========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [13]:
# Train Model
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred_LR = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred_LR)

print("Accuracy of LR :", accuracy)

Accuracy of LR : 0.8472925057661194


In [14]:
model_KN_5 = KNeighborsClassifier(n_neighbors=5)

model_KN_5.fit(X_train, y_train)
# Make predictions
y_pred_KN_5 = model_KN_5.predict(X_test)
accuracy = accuracy_score(y_test, y_pred_KN_5)

print("Accuracy of KN Value 5 :", accuracy)

Accuracy of KN Value 5 : 0.8377913181176633


In [15]:
model_NB = GaussianNB()

model_NB.fit(X_train, y_train)
y_pred_NB = model_NB.predict(X_test)
accuracy = accuracy_score(y_test, y_pred_NB)

print("Accuracy of NB :", accuracy)

Accuracy of NB : 0.6562360150091225


In [16]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Store your trained models here
models = {
    "Logistic Regression": model,
    "K-Nearest Neighbors (KNN)": model_KN_5,
    "Naive Bayes": model_NB
}

results = []

for name, model in models.items():
    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    results.append({
        "Algorithm": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

# Create DataFrame
comparison_df = pd.DataFrame(results)

# Convert to percentage format
for col in ["Accuracy", "Precision", "Recall", "F1 Score"]:
    comparison_df[col] = comparison_df[col].apply(lambda x: f"{x*100:.2f}%")

# Display table
print(comparison_df.to_markdown(index=False))

| Algorithm                 | Accuracy   | Precision   | Recall   | F1 Score   |
|:--------------------------|:-----------|:------------|:---------|:-----------|
| Logistic Regression       | 84.73%     | 83.76%      | 84.73%   | 83.61%     |
| K-Nearest Neighbors (KNN) | 83.78%     | 82.63%      | 83.78%   | 82.65%     |
| Naive Bayes               | 65.62%     | 76.94%      | 65.62%   | 68.46%     |


In [17]:
# Logistic Regression Algorithm Performs well.

#Reason:
#The relationship between features and the target is approximately linear.
#The model generalizes well.
#The classes are reasonably separable.

In [19]:
import joblib
# Save Files
joblib.dump(model, "LR_Weather_Predictor.pkl")
joblib.dump(Scaler, "scaler_Weather_Predictor.pkl")
joblib.dump(encoded_cols, "columns_Weather_Predictor.pkl")
print("Saved Successfully")

Saved Successfully
